# Time series 6: SARIMA only

Fit **SARIMA(p,d,q)(P,D,Q,s)** per frequency band with **seasonal period s=24** (daily). Train on data before the test set, then **rolling 1-step-ahead** forecast on the test set (last 8760 hours). Same evaluation: MAE and MASE vs naive.

Note: SARIMA with s=24 can be slow to fit; we use a small seasonal order. Try (1,0,1)(1,0,1,24) or (0,0,1)(0,0,1,24) for speed.

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

_root = Path.cwd().resolve()
if _root.name == "time_series":
    _root = _root.parent
DATA_PATH = _root / "data" / "transformed" / "transformed_data.parquet"
if not DATA_PATH.exists():
    DATA_PATH = DATA_PATH.with_suffix(".csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Transform data not found at {DATA_PATH}. Run task transform first.")

df = pd.read_parquet(DATA_PATH) if DATA_PATH.suffix == ".parquet" else pd.read_csv(DATA_PATH)
df = df.sort_values(["date", "hour"]).reset_index(drop=True)
df["datetime"] = pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h")

freq_bands = sorted([c for c in df.columns if c not in ("date", "hour", "datetime")])
TEST_STEPS = 24 * 365
SEASONAL_PERIOD = 24  # daily seasonality

# Set True to run SARIMA on one band only (quick results); False for all bands
RUN_SINGLE_BAND = True
bands_to_run = freq_bands[:1] if RUN_SINGLE_BAND else freq_bands

print(f"Loaded {len(df)} rows. Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Frequency bands: {len(freq_bands)}")
print(f"SARIMA will run on: {len(bands_to_run)} band(s)" + (f" ({bands_to_run[0]})" if RUN_SINGLE_BAND else ""))
print(f"Test window: last {TEST_STEPS} hours (1-step-ahead)")
print(f"SARIMA seasonal period: s = {SEASONAL_PERIOD}")

Loaded 9192 rows. Date range: 2025-01-01 to 2026-01-18
Frequency bands: 70
SARIMA will run on: 1 band(s) (3.100-3.105)
Test window: last 8760 hours (1-step-ahead)
SARIMA seasonal period: s = 24


## Fit SARIMA per band, rolling 1-step forecast on test

SARIMA(p,d,q)(P,D,Q,s): order=(p,d,q), seasonal_order=(P,D,Q,s). We use (1,0,1)(1,0,1,24) so the model learns both non-seasonal and daily seasonal structure.

**Quick run:** Set `RUN_SINGLE_BAND = True` in the cell above to fit only the first frequency band and get results quickly. Set to `False` to run all bands.

In [8]:
import sys
ORDER = (1, 0, 1)  # (p, d, q)
SEASONAL_ORDER = (1, 0, 1, SEASONAL_PERIOD)  # (P, D, Q, s)
rows_sarima = []
n_bands = len(bands_to_run)
for bi, freq in enumerate(bands_to_run):
    print(f"[SARIMA] Band {bi+1}/{n_bands}: {freq} ...", flush=True)
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 2 * SEASONAL_PERIOD + 50:
        print(f"  Skip: not enough data (need {TEST_STEPS + 2 * SEASONAL_PERIOD + 50})", flush=True)
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    train_vals = vals[:-TEST_STEPS]
    test_vals = vals[-TEST_STEPS:]
    test_dts = dts[-TEST_STEPS:]
    try:
        model = SARIMAX(train_vals, order=ORDER, seasonal_order=SEASONAL_ORDER, enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit(disp=False)
    except Exception as e:
        print(f"  Skip: fit failed ({e})", flush=True)
        continue
    print(f"  Fit OK, rolling 1-step forecast ({TEST_STEPS} steps) ...", flush=True)
    for i in range(TEST_STEPS):
        try:
            pred = res.forecast(steps=1)[0]
            if i >= 1:
                rows_sarima.append({"datetime": test_dts[i], "frequency_band": freq, "actual": float(test_vals[i]), "predicted": float(pred)})
            res = res.append([test_vals[i]], refit=False)
        except Exception as e:
            print(f"  Forecast broke at step {i}: {e}", flush=True)
            break
        if (i + 1) % 2000 == 0:
            print(f"    step {i+1}/{TEST_STEPS}", flush=True)
    print(f"  Done band {bi+1}/{n_bands}.", flush=True)

df_sarima = pd.DataFrame(rows_sarima)
print(f"[SARIMA] Total: {len(df_sarima)} predictions")

[SARIMA] Band 1/1: 3.100-3.105 ...
  Fit OK, rolling 1-step forecast (8760 steps) ...
    step 2000/8760
    step 4000/8760
    step 6000/8760
    step 8000/8760
  Done band 1/1.
[SARIMA] Total: 8759 predictions


## Naive baseline and MASE

In [9]:
rows_naive = []
for freq in freq_bands:
    sub = df[["datetime", freq]].dropna(subset=[freq]).sort_values("datetime").reset_index(drop=True)
    if len(sub) < TEST_STEPS + 1:
        continue
    vals = sub[freq].values.astype(np.float64)
    dts = sub["datetime"].values
    test_vals, test_dts = vals[-TEST_STEPS:], dts[-TEST_STEPS:]
    for i in range(0, TEST_STEPS - 1):
        rows_naive.append({"datetime": test_dts[i+1], "frequency_band": freq, "actual": test_vals[i+1], "predicted": test_vals[i]})
df_naive = pd.DataFrame(rows_naive)

def mase_from_df(pred_df):
    diffs = pred_df.sort_values(["frequency_band", "datetime"]).groupby("frequency_band")["actual"].diff().dropna().abs()
    denom = float(diffs.mean()) if len(diffs) > 0 else np.nan
    mae = float(np.mean(np.abs(pred_df["actual"] - pred_df["predicted"])))
    return mae / denom if denom and denom > 0 else np.nan

mae_naive = float(np.mean(np.abs(df_naive["actual"] - df_naive["predicted"])))
mase_naive = mase_from_df(df_naive)
print(f"Naive:  MAE = {mae_naive:.4f},  MASE = {mase_naive:.4f}" if not np.isnan(mase_naive) else f"Naive:  MAE = {mae_naive:.4f}")

Naive:  MAE = 2.4198,  MASE = 0.9999


## SARIMA vs Naive

In [10]:
if len(df_sarima) == 0:
    print("No SARIMA predictions (fit failed for all bands). Try smaller order or (0,0,1)(0,0,1,24).")
else:
    mae_sarima = float(np.mean(np.abs(df_sarima["actual"] - df_sarima["predicted"])))
    mase_sarima = mase_from_df(df_sarima)
    # When running single band, compare naive on same band(s) for fair comparison
    naive_compare = df_naive[df_naive["frequency_band"].isin(bands_to_run)]
    mae_naive_compare = float(np.mean(np.abs(naive_compare["actual"] - naive_compare["predicted"]))) if len(naive_compare) > 0 else mae_naive
    mase_naive_compare = mase_from_df(naive_compare) if len(naive_compare) > 0 else mase_naive
    print("=== 1-step-ahead: SARIMA vs Naive ===" + (" (same band(s) only)" if RUN_SINGLE_BAND else ""))
    print(f"  Naive:   MAE = {mae_naive_compare:.4f},  MASE = {mase_naive_compare:.4f}" if not np.isnan(mase_naive_compare) else f"  Naive:   MAE = {mae_naive_compare:.4f}")
    print(f"  SARIMA:  MAE = {mae_sarima:.4f},  MASE = {mase_sarima:.4f}" if not np.isnan(mase_sarima) else f"  SARIMA:  MAE = {mae_sarima:.4f}")
    if mae_sarima < mae_naive_compare:
        print("  → SARIMA beats naive.")
    else:
        print("  → Naive best (common at 1-step for very persistent series).")

=== 1-step-ahead: SARIMA vs Naive === (same band(s) only)
  Naive:   MAE = 3.5300,  MASE = 1.0000
  SARIMA:  MAE = 10.6354,  MASE = 3.0127
  → Naive best (common at 1-step for very persistent series).
